# CSV Processing for Readability

This notebook processes the csv files produced from the random search so that it includes interesting relevant properties such as unimodality and defectivity.

**Warning:** One should be cautious to check if an architecture is truly minimal. The column "is_minimal" is in reference to the search *thus far*. It is possible for it to actually not be minimal and filling because the search has not uncovered a subarchitecture that is filling yet. One should manually check minimality by computing the dimension of the subarchitectures.

In [1]:
import pandas as pd
import math
import ast
import os

## Helper Functions

In [2]:
# %load ../is_unimodal.py

def is_unimodal(data):
    # data should be string formatted as '[1, 2, 3]'
    data = ast.literal_eval(data)
    n = len(data)
    
    if n <= 2:
        return True
    
    i = 0
    
    # walk up the non-decreasing slope
    while i + 1 < n and data[i] <= data[i + 1]:
        i += 1

    # walk down the non-increasing slope
    while i + 1 < n and data[i] >= data[i + 1]:
        i += 1
        
    # check if we reached the end
    return i == n - 1

In [3]:
def expdim(d: list[int], r) -> int | float:
    """
    Calculates the expected dimension of the architecture
    """

    if type(d)==str:
        d = ast.literal_eval(d)
    
    h = len(d) - 1
    d_0 = d[0]
    d_h = d[h]

    # Term 1: d_h + sum_{i=1}^{h} (d_{i-1}-1)*d_i
    sum_term = sum((d[i-1]-1)*d[i] for i in range(1,h+1))
    sum_term += d_h

    # Term 2: d_h * Binomial(r**{h-1} + d_0 - 1, d_0 - 1)
    n = r**(h-1) + d_0 - 1
    k = d_0 - 1
    
    ambient_term = d_h * math.comb(n, k)

    return min(sum_term, ambient_term)

## Processing

In [4]:
ARCHS = [(2,1,2), (2,2,2), (2,3,2)]

for entry in ARCHS:
    d0 = entry[0]
    dh = entry[1]
    r = entry[2]

    df = pd.read_csv(f'../data/raw/{d0}_{dh}_architectures.csv')

    output_file = f'../data/processed/{d0}_{dh}_cleaned_architectures.csv'

    df['is_unimodal'] = df['architecture'].apply(is_unimodal)
    df['expected_dimension'] = df['architecture'].apply(lambda x: expdim(x, r))
    df['defect'] = df['expected_dimension'] - df['dimension_computed'] 
    df = df.sort_values(by=['h','is_minimal','is_unimodal'], ascending=[True,False,True])

    print(f"For d_0={d0} and d_h={dh}: the following are counterexamples to the minimal unimodal conjecture.")
    display(df[(df['is_minimal']==True) & (df['is_full_dimension']==True) & (df['is_unimodal']==False)])

    df.to_csv(output_file, index=False, header=f"Architectures with d_0={d0} and d_h={dh}")

For d_0=2 and d_h=1: the following are counterexamples to the minimal unimodal conjecture.


,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,is_unimodal,expected_dimension,defect
523,7,2,"[2, 3, 4, 5, 4, 6, 4, 1]",110,65,65,True,True,False,65,0
5886,8,2,"[2, 3, 4, 5, 4, 8, 8, 4, 1]",190,129,129,True,True,False,129,0
5937,8,2,"[2, 3, 4, 5, 6, 5, 8, 5, 1]",183,129,129,True,True,False,129,0
6044,8,2,"[2, 3, 4, 5, 7, 6, 7, 3, 1]",181,129,129,True,True,False,129,0
6117,8,2,"[2, 3, 3, 5, 7, 6, 8, 3, 1]",182,129,129,True,True,False,129,0
6277,8,2,"[2, 3, 4, 4, 7, 6, 8, 3, 1]",179,129,129,True,True,False,129,0
6303,8,2,"[2, 3, 4, 5, 7, 5, 7, 5, 1]",183,129,129,True,True,False,129,0
6305,8,2,"[2, 3, 4, 6, 6, 5, 7, 7, 1]",199,129,129,True,True,False,129,0
6308,8,2,"[2, 3, 3, 6, 7, 6, 7, 3, 1]",183,129,129,True,True,False,129,0
6311,8,2,"[2, 3, 4, 4, 7, 6, 7, 5, 1]",186,129,129,True,True,False,129,0


For d_0=2 and d_h=2: the following are counterexamples to the minimal unimodal conjecture.


,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,is_unimodal,expected_dimension,defect
14834,9,2,"[2, 3, 4, 4, 10, 17, 11, 12, 4, 2]",619,514,514,True,True,False,514,0


For d_0=2 and d_h=3: the following are counterexamples to the minimal unimodal conjecture.


,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,is_unimodal,expected_dimension,defect
8950,7,2,"[2, 3, 4, 7, 8, 7, 10, 3]",258,195,195,True,True,False,195,0
9076,7,2,"[2, 3, 4, 5, 9, 7, 10, 3]",246,195,195,True,True,False,195,0


In [5]:
ARCHS = [(2,1,2), (2,2,2), (2,3,2)]

for entry in ARCHS:
    d0 = entry[0]
    dh = entry[1]
    r = entry[2]

    df = pd.read_csv(f'../data/raw/{d0}_{dh}_architectures.csv')

    output_file = f'../data/processed/{d0}_{dh}_cleaned_architectures.csv'

    df['is_unimodal'] = df['architecture'].apply(is_unimodal)
    df['expected_dimension'] = df['architecture'].apply(lambda x: expdim(x, r))
    df['defect'] = df['expected_dimension'] - df['dimension_computed'] 
    df = df.sort_values(by=['h','is_minimal','is_unimodal'], ascending=[True,False,True])

# CSV Of Counterexamples

Below, we extra the list of counterexamples to the minimal unimodal conjecture and save it to a separate .csv file.

In [6]:
import pandas as pd

ARCHS = [(2,1,2), (2,2,2), (2,3,2)]

ce = pd.DataFrame()

for entry in ARCHS:

    d0, dh, r = entry

    df = pd.read_csv(f'../data/raw/{d0}_{dh}_architectures.csv')

    df['is_unimodal'] = df['architecture'].apply(is_unimodal)
    df['expected_dimension'] = df['architecture'].apply(lambda x: expdim(x, r))
    df['defect'] = df['expected_dimension'] - df['dimension_computed'] 
    
    df['d0'] = d0
    df['dh'] = dh
    df['r'] = r

    df = df.sort_values(by=['h','is_minimal','is_unimodal'], ascending=[True,False,True])

    # Filter for counterexamples (fixed the missing closing brackets)
    counterexamples = df[(df['is_minimal'] == True) & 
                         (df['is_full_dimension'] == True) & 
                         (df['is_unimodal'] == False)]
    
    # Save/append to the master CE dataframe
    if not counterexamples.empty:
        ce = pd.concat([ce, counterexamples], ignore_index=True)

ce.to_csv('../data/processed/counterexamples.csv', index=False)
display(ce)

,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,is_unimodal,expected_dimension,defect,d0,dh,r
0,7,2,"[2, 3, 4, 5, 4, 6, 4, 1]",110,65,65,True,True,False,65,0,2,1,2
1,8,2,"[2, 3, 4, 5, 4, 8, 8, 4, 1]",190,129,129,True,True,False,129,0,2,1,2
2,8,2,"[2, 3, 4, 5, 6, 5, 8, 5, 1]",183,129,129,True,True,False,129,0,2,1,2
3,8,2,"[2, 3, 4, 5, 7, 6, 7, 3, 1]",181,129,129,True,True,False,129,0,2,1,2
4,8,2,"[2, 3, 3, 5, 7, 6, 8, 3, 1]",182,129,129,True,True,False,129,0,2,1,2
5,8,2,"[2, 3, 4, 4, 7, 6, 8, 3, 1]",179,129,129,True,True,False,129,0,2,1,2
6,8,2,"[2, 3, 4, 5, 7, 5, 7, 5, 1]",183,129,129,True,True,False,129,0,2,1,2
7,8,2,"[2, 3, 4, 6, 6, 5, 7, 7, 1]",199,129,129,True,True,False,129,0,2,1,2
8,8,2,"[2, 3, 3, 6, 7, 6, 7, 3, 1]",183,129,129,True,True,False,129,0,2,1,2
9,8,2,"[2, 3, 4, 4, 7, 6, 7, 5, 1]",186,129,129,True,True,False,129,0,2,1,2
